# Explainability - part 2: Grad-CAM, and whether to trust it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/explainability/notebooks/ex3_gradcam.ipynb)

**Computer-Aided Medical Diagnostics (MPA-PRM)** · lab for the lecture
*Explainability and interpretability of models*

A ResNet18 fine-tuned on PneumoniaMNIST chest X-rays, Grad-CAM written by
hand, and then the trust exercise: destroy part of the model and see which
explanation notices.

The training takes a few minutes on a CPU; with *Runtime - Change runtime
type - T4 GPU* it takes seconds.

**Three holes.** The fine-tuning, Grad-CAM, and the randomized copy of the
model.

# Data

PneumoniaMNIST (Yang et al., *MedMNIST v2*, CC BY 4.0): 28 × 28 chest X-rays
of children, normal or pneumonia, downloaded from Zenodo (4.2 MB).

In [ ]:
import hashlib
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEVICE)

URL = "https://zenodo.org/records/10519652/files/pneumoniamnist.npz?download=1"
MD5 = "28209eda62fecd6e6a2d98b1501bb15f"
SEED = 42
N_TRAIN = 1500
N_EPOCHS = 3
CLASSES = ["normal", "pneumonia"]


def download(path=Path("pneumoniamnist.npz")):
    """Downloads the 4.2 MB PneumoniaMNIST archive once and checks it."""
    if not path.is_file():
        urllib.request.urlretrieve(URL, path)
    if hashlib.md5(path.read_bytes()).hexdigest() != MD5:
        raise RuntimeError(f"{path} is damaged - delete it and run again")
    return path


def load_pneumonia(path: Path | None = None) -> dict[str, np.ndarray]:
    """Images as float32 arrays (N, 28, 28) in [0, 1], labels as int64 (N,)."""
    archive = np.load(download() if path is None else path)
    data = {}
    for split in ("train", "val", "test"):
        data[f"x_{split}"] = archive[f"{split}_images"].astype(np.float32) / 255.0
        data[f"y_{split}"] = archive[f"{split}_labels"].reshape(-1).astype(np.int64)
    return data


def to_input(images: np.ndarray) -> torch.Tensor:
    """(N, 28, 28) in [0, 1] -> the network input (N, 3, 224, 224), normalized to [-1, 1]."""
    x = torch.from_numpy(np.ascontiguousarray(images, dtype=np.float32)).unsqueeze(1)
    x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
    return ((x - 0.5) / 0.5).repeat(1, 3, 1, 1)


def predict(model: nn.Module, images: np.ndarray, device: str = "cpu") -> np.ndarray:
    """Probability of pneumonia for every image, in batches of 64."""
    probabilities = []
    with torch.inference_mode():
        for start in range(0, len(images), 64):
            logits = model(to_input(images[start:start + 64]).to(device))
            probabilities.append(logits.softmax(1)[:, 1].cpu().numpy())
    return np.concatenate(probabilities)


data = load_pneumonia()
for split in ("train", "val", "test"):
    print(f"{split:5s} {len(data['y_' + split]):5d} X-rays, "
          f"{data['y_' + split].mean():.0%} pneumonia")

# The model

### Hole 1 - fine-tune a ResNet18

In [ ]:
# TODO 1: fine-tune an ImageNet ResNet18 - the shared recipe.
#
#   torch.manual_seed(SEED); a random subset of N_TRAIN = 1500 training
#   images (np.random.RandomState(SEED).choice(..., replace=False));
#   tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1) with
#   model.fc replaced by nn.Linear(512, 2); Adam with lr 1e-4; N_EPOCHS = 3
#   epochs over batches of 32 in a random order (torch.randperm with a
#   generator seeded by SEED); the loss F.cross_entropy(model(to_input(x)), y).
#   Return the model in eval() mode.

def train_resnet(x_train, y_train, n_train=N_TRAIN, n_epochs=N_EPOCHS, device="cpu", verbose=True):
    ...


model = train_resnet(data["x_train"], data["y_train"], device=DEVICE)

In [ ]:
model = model.cpu()   # the explanations below run on the CPU
p_test = predict(model, data["x_test"])
print(f"test accuracy, all 624 X-rays: {((p_test > 0.5) == data['y_test']).mean():.3f}")

# Grad-CAM

The last convolutional block of the ResNet18, `layer4[-1]`, gives 512 maps of
7 × 7. Grad-CAM weighs each map by how much the class score depends on it,
sums them, and keeps the positive part.

### Hole 2 - Grad-CAM in fifteen lines

In [ ]:
# TODO 2: Grad-CAM by hand - the two formulas of the lecture.
#
#   1) a forward hook on model.layer4[-1] stores its output A (1, 512, 7, 7):
#        hook = model.layer4[-1].register_forward_hook(
#            lambda module, inputs, output: stored.update(A=output))
#   2) the score of the target class: model(to_input(image[None]))[0, target];
#      remove the hook;
#   3) its gradient with respect to A: torch.autograd.grad(score, A);
#   4) alpha = the gradient averaged over the 7 x 7 positions (one number per map);
#   5) cam = ReLU(sum over maps of alpha * A), upsampled to 224 x 224
#      (F.interpolate, bilinear) and divided by its maximum.

def grad_cam(model, image, target):
    stored = {}
    ...

In [ ]:
def overlay(ax, image, heat):
    ax.imshow(image, cmap="gray", vmin=0, vmax=1, extent=(0, 1, 1, 0))
    ax.imshow(heat, cmap="jet", alpha=0.45, extent=(0, 1, 1, 0))
    ax.axis("off")


rng = np.random.RandomState(SEED)
shown = rng.choice(len(data["x_test"]), size=6, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for col, i in enumerate(shown):
    image, label = data["x_test"][i], data["y_test"][i]
    target = int(p_test[i] > 0.5)
    axes[0, col].imshow(image, cmap="gray", vmin=0, vmax=1)
    axes[0, col].set_title(f"true: {CLASSES[label]}", fontsize=9)
    axes[0, col].axis("off")
    overlay(axes[1, col], image, grad_cam(model, image, target))
    mark = "OK" if target == label else "WRONG"
    axes[1, col].set_title(f"{mark} {CLASSES[target]} ({max(p_test[i], 1 - p_test[i]):.0%})", fontsize=9)
plt.show()

Are the highlighted regions inside the lung fields? Find a wrong
prediction - does its map suggest why?

# The trust exercise

Adebayo et al. (2018): an explanation that does not change when the model is
destroyed was never describing the model.

### Hole 3 - a randomized copy

In [ ]:
# TODO 3: the sanity check of Adebayo et al. A deep copy of the model
#         (copy.deepcopy) with layer4 and the classifier head re-initialized:
#         for every module in list(random.layer4.modules()) + [random.fc]
#         call module.reset_parameters() if it has one, and
#         module.reset_running_stats() if it is an nn.BatchNorm2d.
#         Seed with torch.manual_seed(SEED) first; return it in eval() mode.
import copy


def randomized_copy(model, seed=SEED):
    ...


random_model = randomized_copy(model)
print(f"randomized model, test accuracy "
      f"{((predict(random_model, data['x_test']) > 0.5) == data['y_test']).mean():.3f}")

In [ ]:
# The quiz: three X-rays, two Grad-CAM maps each - one from the trained
# model, one from the randomized copy, in a random order. Decide which one is
# real BEFORE you run the next cell.
rng = np.random.RandomState(7)
quiz = rng.choice(len(data["x_test"]), size=3, replace=False)
real_is_A = []
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for row, i in enumerate(quiz):
    image = data["x_test"][i]
    target = int(p_test[i] > 0.5)
    real, fake = grad_cam(model, image, target), grad_cam(random_model, image, target)
    real_is_A.append(bool(rng.randint(2)))
    first, second = (real, fake) if real_is_A[-1] else (fake, real)
    axes[row, 0].imshow(image, cmap="gray", vmin=0, vmax=1); axes[row, 0].axis("off")
    overlay(axes[row, 1], image, first)
    overlay(axes[row, 2], image, second)
axes[0, 0].set_title("input"); axes[0, 1].set_title("A"); axes[0, 2].set_title("B")
plt.show()

Decided? Then reveal.

In [ ]:
for row, a in enumerate(real_is_A):
    print(f"X-ray {row + 1}: the trained model is {'A' if a else 'B'}")

In [ ]:
# The same test for two more methods, measured instead of eyeballed: the
# rank correlation between the map of the trained and of the randomized
# model, over 20 test X-rays (1 = identical maps, 0 = unrelated).
from scipy.stats import spearmanr


def to_input_differentiable(x: torch.Tensor) -> torch.Tensor:
    """`to_input` for a (N, 28, 28) tensor that carries a gradient."""
    x = F.interpolate(x.unsqueeze(1), size=(224, 224), mode="bilinear", align_corners=False)
    return ((x - 0.5) / 0.5).repeat(1, 3, 1, 1)


def vanilla_gradient(model: nn.Module, image: np.ndarray, target: int) -> np.ndarray:
    """|d y_target / d x| on the 28 x 28 image, scaled to [0, 1]."""
    x = torch.from_numpy(image[None].astype(np.float32)).requires_grad_(True)
    score = model(to_input_differentiable(x))[0, target]
    gradient, = torch.autograd.grad(score, x)
    saliency = gradient[0].abs().numpy()
    return saliency / saliency.max() if saliency.max() > 0 else saliency


class _GuidedReLU(torch.autograd.Function):
    """A ReLU whose backward pass lets through only positive gradients at
    positive inputs - the whole of guided backpropagation."""

    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * (x > 0) * (grad_output > 0)


class _GuidedReLUModule(nn.Module):
    def forward(self, x):
        return _GuidedReLU.apply(x)


def _guided(model: nn.Module) -> nn.Module:
    """A copy of the model with every ReLU replaced by the guided one."""
    guided = copy.deepcopy(model)
    for module in list(guided.modules()):
        for name, child in module.named_children():
            if isinstance(child, nn.ReLU):
                setattr(module, name, _GuidedReLUModule())
    return guided.eval()


def guided_backprop(model: nn.Module, image: np.ndarray, target: int) -> np.ndarray:
    """|guided gradient| on the 28 x 28 image, scaled to [0, 1]."""
    return vanilla_gradient(_guided(model), image, target)


sample = np.random.RandomState(0).choice(len(data["x_test"]), 20, replace=False)
for name, explain in [("Grad-CAM", grad_cam), ("vanilla gradient", vanilla_gradient),
                      ("guided backprop", guided_backprop)]:
    correlations = []
    for i in sample:
        target = int(p_test[i] > 0.5)
        a = explain(model, data["x_test"][i], target)
        b = explain(random_model, data["x_test"][i], target)
        correlations.append(spearmanr(a.ravel(), b.ravel()).statistic)
    print(f"{name:17s} rank correlation trained vs. randomized: {np.mean(correlations):.3f}")

# Questions

1. Could you tell the real map before the reveal? What does that say about
   the evidence a single heatmap gives?
2. Grad-CAM and the gradient changed with the model, guided backprop did not.
   What is guided backprop showing, then?
3. Would you trust a clinical system that shows clinicians a heatmap with no
   sanity check behind it? What would you want to see first?

---

*The numbers are computed by [`pneumonia.py`](https://github.com/tomasvicar/AUI-public/blob/master/lectures/explainability/code/pneumonia.py)
of the lecture. The notebook is built by
[`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/explainability/code/build_notebooks.py) - editing the
`.ipynb` by hand gets overwritten.*